In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
# Load Diabetes dataset
diabetes = load_diabetes()

X = diabetes.data
y = diabetes.target

print("Dataset shape:", X.shape)
print("Features:", diabetes.feature_names)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

Dataset shape: (442, 10)
Features: ['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']


In [3]:
linear_model = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression())
])

linear_model.fit(X_train, y_train)

y_pred_linear = linear_model.predict(X_test)

linear_mse = mean_squared_error(y_test, y_pred_linear)
linear_r2 = r2_score(y_test, y_pred_linear)

print("Linear Regression")
print("MSE:", linear_mse)
print("R²:", linear_r2)

Linear Regression
MSE: 2900.193628493482
R²: 0.4526027629719195


In [4]:
ridge_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])

ridge_params = {
    "ridge__alpha": np.logspace(-3, 3, 20)
}

ridge_grid = GridSearchCV(
    ridge_pipeline,
    ridge_params,
    cv=5,
    scoring="neg_mean_squared_error"
)

ridge_grid.fit(X_train, y_train)

best_ridge = ridge_grid.best_estimator_

y_pred_ridge = best_ridge.predict(X_test)

ridge_mse = mean_squared_error(y_test, y_pred_ridge)
ridge_r2 = r2_score(y_test, y_pred_ridge)

print("Best Ridge alpha:", ridge_grid.best_params_["ridge__alpha"])
print("Ridge MSE:", ridge_mse)
print("Ridge R²:", ridge_r2)

Best Ridge alpha: 26.366508987303554
Ridge MSE: 2864.1356084929694
Ridge R²: 0.45940853632686496


In [5]:
lasso_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso", Lasso(max_iter=10000))
])

lasso_params = {
    "lasso__alpha": np.logspace(-3, 2, 20)
}

lasso_grid = GridSearchCV(
    lasso_pipeline,
    lasso_params,
    cv=5,
    scoring="neg_mean_squared_error"
)

lasso_grid.fit(X_train, y_train)

best_lasso = lasso_grid.best_estimator_

y_pred_lasso = best_lasso.predict(X_test)

lasso_mse = mean_squared_error(y_test, y_pred_lasso)
lasso_r2 = r2_score(y_test, y_pred_lasso)

print("Best Lasso alpha:", lasso_grid.best_params_["lasso__alpha"])
print("Lasso MSE:", lasso_mse)
print("Lasso R²:", lasso_r2)

Best Lasso alpha: 1.438449888287663
Lasso MSE: 2803.892804814914
Lasso R²: 0.470779068266597


In [6]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Ridge Regression",
        "Lasso Regression"
    ],
    "MSE": [
        linear_mse,
        ridge_mse,
        lasso_mse
    ],
    "R-Squared": [
        linear_r2,
        ridge_r2,
        lasso_r2
    ]
})

print(results)

               Model          MSE  R-Squared
0  Linear Regression  2900.193628   0.452603
1   Ridge Regression  2864.135608   0.459409
2   Lasso Regression  2803.892805   0.470779


In [7]:
print("Lasso Coefficients:")

coefficients = best_lasso.named_steps["lasso"].coef_

for feature, coefficient in zip(
    diabetes.feature_names,
    coefficients
):
    print(f"{feature}: {coefficient:.4f}")

Lasso Coefficients:
age: 0.1154
sex: -8.3441
bmi: 26.2126
bp: 15.2131
s1: -5.2854
s2: -0.0000
s3: -11.2086
s4: 0.0000
s5: 22.2407
s6: 1.8943
